In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.vector_stores.deeplake import DeepLakeVectorStore

c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.5.9) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [2]:
import os
import groq
groq.api_key = os.getenv("GROQ_API_KEY")

In [3]:
from pathlib import Path
data_dir = Path(os.path.join(os.getcwd(), 'data'))
data_dir.mkdir(exist_ok=True)
print(data_dir)

c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch3\data


**Pipeline 1: Collecting and preparing the documents**

In [4]:
import requests
from bs4 import BeautifulSoup
import re
import os

urls = [
    "https://github.com/VisDrone/VisDrone-Dataset",
    "https://paperswithcode.com/dataset/visdrone",
    "https://openaccess.thecvf.com/content_ECCVW_2018/papers/11133/Zhu_VisDrone-DET2018_The_Vision_Meets_Drone_Object_Detection_in_Image_Challenge_ECCVW_2018_paper.pdf",
    "https://github.com/VisDrone/VisDrone2018-MOT-toolkit",
    "https://en.wikipedia.org/wiki/Object_detection",
    "https://en.wikipedia.org/wiki/Computer_vision",
    "https://en.wikipedia.org/wiki/Convolutional_neural_network",
    "https://en.wikipedia.org/wiki/Unmanned_aerial_vehicle",
    "https://www.faa.gov/uas/",
    "https://www.tensorflow.org/",
    "https://pytorch.org/", 
    "https://keras.io/",
    "https://arxiv.org/abs/1804.06985",
    "https://arxiv.org/abs/2202.11983",
    "https://www.dronedeploy.com/",
]

The corpus contains a list of sites related to CV and drones. However, the list contains also noisy links. In this case, we are working with unstructured data in various formats and variable quality.

In [5]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; RAGBasicsBot/2.0)'}

def clean_text(text):
    #Remove references like [1], [2], etc..
    content = re.sub(r'\[\d+\]', '', text)
    content = re.sub(r'[^\w\s\.]', '', content)  # Remove punctuation (except periods)
    return content

def fetch_and_clean(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        for tag in soup(['script', 'style']):
            tag.decompose()
        
        main_content = None
        main_content = soup.find('div', {'class': 'mw-parser-output'})
        if not main_content:
            main_content = soup.find('div', {'class': 'content'})
        if not main_content:
            main_content = soup.find('div', {'class': 'article-content'}) or \
                          soup.find('div', {'class': 'post-content'}) or \
                          soup.find('main') or \
                          soup.find('article')
        
        if not main_content:
            main_content = soup.find('body')
        if not main_content:
            print(f"Warning: could not find main content for {url}, skipping.")
            return ''
        for section_title in ['References', 'Bibliography', 'External links', 'See also', 'Notes', 'Related links']:
            for section in main_content.find_all('span', id=section_title):
                for sib in section.parent.find_next_siblings():
                    sib.decompose()
                section.parent.decompose()
        
        text = main_content.get_text(separator=' ', strip=True)
        text = clean_text(text)
        return text if text else ''
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return ''

Each project will reuqire specific names and paths from original data

In [6]:
docs_dir = data_dir / 'documents'
docs_dir.mkdir(exist_ok=True)
def download_docs(urls):
    for url in urls:
        clean_text = fetch_and_clean(url)
        if clean_text is None:
            print(f"No content for {url}, skipping.")
            continue
        doc_split = url.split('/')
        doc_name = doc_split[-1] if doc_split[-1] != '' else doc_split[-2]
        filename = os.path.join(docs_dir, doc_name + '.txt')
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(clean_text)
    
download_docs(urls)


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Error fetching https://www.faa.gov/uas/: 403 Client Error: Forbidden for url: https://www.faa.gov/uas/


In [7]:
docs = SimpleDirectoryReader(docs_dir).load_data()
docs[0]

Document(id_='c4328e56-f2b4-4d8c-9520-a38bbd90c6f3', embedding=None, metadata={'file_path': 'c:\\Users\\dorot\\OneDrive - Politecnico di Torino\\Desktop\\Progetti\\RAG_Books\\RAG Driven Generative AI\\Ch3\\data\\documents\\1804.06985.txt', 'file_name': '1804.06985.txt', 'file_type': 'text/plain', 'file_size': 3782, 'creation_date': '2026-03-25', 'last_modified_date': '2026-03-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='High Energy Physics  Theory arXiv1804.06985 hepth Submitted on 19 Apr 2018 Title A Near Horizon Extreme Binary Black Hole Geometry Authors Jacob Ciafre  Maria J. Rodriguez View a PDF of the paper titled A Near Hori

The SimplyDirectoryReader is designed for working with unstructured data. It recursively scans teh directory and identifies and loads all supported file types (txt, docx...). It then extracts teh content from each file and returns a list of document objects with its text and metadata, such as teh filename and filepath. 

**Pipeline 2: creating and populating a Deep lake vector store**

We will create a Deep Lake vector store and populate it with data in our documents. We will implement a standard tensor configuration with:
- text(str): the text is the content of one of the text files listed in the dictionary of documents.
- metadata(json): will contain the filename source of each chunk of text for full transparency and control.
- embedding(float32)
- id(str, auto-populated): a unique id is attributed automatically to each chunk. the vector store will also contain an index, which is  a number from 0 to n, but it cannot be used semantically since it will change each time we modify the dataset. However, the id field will remain unchanged until we decide to optimize it with index-based search strategies.

In [8]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

2026-03-25 12:48:41,669 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


In [11]:
from llama_index.core import StorageContext
from llama_index.vector_stores.deeplake import DeepLakeVectorStore
from llama_index.core import VectorStoreIndex

vector_store_path = str(data_dir / "dory_vector_store")

vector_store = DeepLakeVectorStore(
    dataset_path=vector_store_path, overwrite=True
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [12]:
index = VectorStoreIndex.from_documents(documents=docs, storage_context=storage_context, embed_model=embed_model)

Uploading data to deeplake dataset.


  0%|          | 0/120 [00:00<?, ?it/s]


Exception: Error while attempting to rollback appends

In [14]:
import deeplake
ds = deeplake.load(vector_store_path)

c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch3\data\dory_vector_store loaded successfully.



In [15]:
import json
import pandas as pd
import numpy as np

data = {}

for tensor_name in ds.tensors:
    tensor_data = ds[tensor_name].numpy()
    if tensor_data.ndim > 1:
        data[tensor_name] = [np.array(e).flatten().tolist() for e in tensor_data]
    else:
        if tensor_name == 'text':
            data[tensor_name] = [t.tobytes().decode('utf-8') for t in tensor_data]
        else:
            data[tensor_name] = tensor_data.tolist()
    
df = pd.DataFrame(data)

ValueError: All arrays must be of the same length

In [ ]:
def display_record(record_number):
    record = df.iloc[record_number]
    display_data = {
        "ID": record['id'] if 'id' in record else None,
        "Text": record['text'] if 'text' in record else None,
        "Metadata": record['metadata'] if 'metadata' in record else None,
        "Embedding": record['embedding'] if 'embedding' in record else None,
    }

In [ ]:
rec = 0
display_record(rec)

IndexError: single positional indexer is out-of-bounds

The id is the index we will be using to organize teh chunks of text of teh text colum in the dataset. The chunks will be transformed into nodes that can contain teh original text, summaries and additional information. 

The metadata was automatically generated by SimpleDirectoryReader.

The text column contains the chunked data.

The embedding of each chunk of datawas generated through an embedding model that we do not need to configure.

**Pipeline 3: Index-based RAG**

We will implement four index engines:
- Vector Store index Engine: creates a vector store index from the documents, enabling efficient similarity-based searches.
- Tree Index: builds a hierarchical tree index from teh documents.
- List Index: constructs a straightfroward idnex from the documents.
- Keyword table Index: creates an index based on keywords extracted from the documents.

We will implementing querying with an LLM: queries the index with user input, retrieves teh relevanmt documents and returns a synthesized response.

In [ ]:
user_input = "What is the VisDrone dataset?"

In [ ]:
k = 3 #number of retrieved documents to use for the generation of the answer
temp = 0.1 #temperature for the generation of the answer
mt = 1024 #number of tokens to use for the retrieved context when generating the answer

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_similarity(text1, text2):
    emb1 = model.encode([text1])
    emb2 = model.encode([text2])
    similarity = cosine_similarity([emb1], [emb2])
    return similarity[0][0]

2026-03-24 18:59:05,598 - INFO - Use pytorch device_name: cuda:0
2026-03-24 18:59:05,599 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


1) Vector Store Index Query Engine
VectorStoreIndex is a type of index within LlamaIndex that implements vector embeddings to represent and retrieve infromation from docs. These docs with similar meanings will have embeddings that are closer together in teh vector space. However, this time, it does not use the existing Deep Lake vector store, it can create a new memory vector index, re-embed the docs and create a new index structure. 

In [ ]:
from llama_index.core import VectorStoreIndex
vector_store_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(type(vector_store_index))

2026-03-24 19:05:05,379 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,381 - INFO - Retrying request to /embeddings in 0.401726 seconds
2026-03-24 19:05:05,870 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,870 - INFO - Retrying request to /embeddings in 0.855444 seconds
2026-03-24 19:05:06,813 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:06,815 - INFO - Retrying request to /embeddings in 1.555540 seconds
2026-03-24 19:05:08,468 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:08,469 - INFO - Retrying request to /embeddings in 3.368408 seconds
2026-03-24 19:05:11,893 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:11,895 - INFO - Retrying request 

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
vector_query_engine = vector_store_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

NameError: name 'vector_store_index' is not defined

This function executes a query using a vector query engine and processes teh results into a structured format.

In [ ]:
import pandas as pd
import textwrap

def index_query(input_query):
    response = vector_query_engine.query(input_query)
    print(textwrap.fill(str(response), width=80))
    node_data =[]
    for node in response.source_nodes:
        noe_info = {
            "id": node.node_id,
            "text": node.node.get_text(),
            "score" : node.score
        }
        node_data.append(noe_info)
    df = pd.DataFrame(node_data)
    return df, response

Optimized chunking: we can decide the chunk size or teh code determines teh chunk size automatically.

In [ ]:
for node in response.source_nodes:
    node = node.node
    chunk_size = len(node.get_text())
    print(f"Node ID: {node.node_id}, Chunk Size: {chunk_size}")


We first calculate the sum of the scores and the average scores, and then we divide the weighted average by the time elapsed to perform the query. This metric is not an absolute value. It's an indicator that we can use to compare this engine with others.



In [ ]:
import numpy as np

def info_metrics(response):
    scores = [node.score for node in response.source_nodes]
    if scores:
        weights = np.exp(scores) / np.sum(np.exp(scores))  # Softmax to get weights
        perf = np.average(scores, weights=weights) / elapsed_time  # Example performance metric 
    else:
        perf = 0
    return perf

2) Tree index query Engine
The tree index organizes docs in a tree structure with broader summaries at higher levels and detailed information lower levels. Each node in the tree summarizes the text it covers. It is efficient for large datasets and queries large collections of docs rapisly by breaking them down into optimized chunks. In this model, the LLM acts like it is answering a multiple-choice question when selecting the best nodes during a query. It analyzes the query, compares it with teh summaries of the current node's children and decides which path to follow to find the most relevant information.

In [ ]:
from llama_index.core import TreeIndex
tree_index = TreeIndex.from_documents(documents)   

In [ ]:
import time
import textwrap

start_time = time.time()
response = tree_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

NameError: name 'tree_query_engine' is not defined

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

3) Last index query engine
The query engine will process the user input and each doc as a prompt for an LLM. The LLm will evaluate the semantic similarity relationship between the docs and the query, thus implicity ranking and selecting teh most relavnt nodes. The selection with an LLM is not rule-based. Nothing is predefined, which means that the selction is prompt-based by combining the user input with a collection of docs. The LLM evaluates each doc in teh list indipendently, assigning a score based on its perceived relevance to teh query. Then, teh top-k docs are retained by the query engine.

In [ ]:
from llam_index.core import ListIndex
list_index = ListIndex.from_documents(documents)

In [ ]:
list_query_engine = list_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

In [ ]:
start_time = time.time()
response = list_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

4) Keyword index query engine
It is designed to extract keywords from your docs and organize them in a table-like structure. This strcuture makes it easier to query and retrieve relevant information based on specific keyowrds or topics.

In [ ]:
from llama_index.core import KeywordTableIndex  
keyword_index = KeywordTableIndex.from_documents(documents)

2026-03-24 19:52:15,799 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:15,803 - INFO - Retrying request to /chat/completions in 0.444031 seconds
2026-03-24 19:52:16,986 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:16,988 - INFO - Retrying request to /chat/completions in 0.867025 seconds
2026-03-24 19:52:18,546 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:18,548 - INFO - Retrying request to /chat/completions in 1.589668 seconds
2026-03-24 19:52:20,861 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:20,866 - WARNING - Retrying llama_index.llms.openai.base.OpenAI._chat in 1 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
data = []
for keyword, doc_ids in keyword_index.keyword_to_doc_ids.items():
    for doc_id in doc_ids:
        data.append({"keyword": keyword, "doc_id": doc_id})

df = pd.DataFrame(data)
df

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.vector_stores.deeplake import DeepLakeVectorStore

c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\deeplake\util\check_latest_version.py:32: UserWarning: A newer version of deeplake (4.5.9) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


In [ ]:
import os
import groq
groq.api_key = os.getenv("GROQ_API_KEY")

In [ ]:
from pathlib import Path
data_dir = Path(os.path.join(os.getcwd(), 'data'))
data_dir.mkdir(exist_ok=True)
print(data_dir)

c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch3\data


**Pipeline 1: Collecting and preparing the documents**

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import os

urls = [
    "https://github.com/VisDrone/VisDrone-Dataset",
    "https://paperswithcode.com/dataset/visdrone",
    "https://openaccess.thecvf.com/content_ECCVW_2018/papers/11133/Zhu_VisDrone-DET2018_The_Vision_Meets_Drone_Object_Detection_in_Image_Challenge_ECCVW_2018_paper.pdf",
    "https://github.com/VisDrone/VisDrone2018-MOT-toolkit",
    "https://en.wikipedia.org/wiki/Object_detection",
    "https://en.wikipedia.org/wiki/Computer_vision",
    "https://en.wikipedia.org/wiki/Convolutional_neural_network",
    "https://en.wikipedia.org/wiki/Unmanned_aerial_vehicle",
    "https://www.faa.gov/uas/",
    "https://www.tensorflow.org/",
    "https://pytorch.org/", 
    "https://keras.io/",
    "https://arxiv.org/abs/1804.06985",
    "https://arxiv.org/abs/2202.11983",
    "https://www.dronedeploy.com/",
]

The corpus contains a list of sites related to CV and drones. However, the list contains also noisy links. In this case, we are working with unstructured data in various formats and variable quality.

In [ ]:
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; RAGBasicsBot/2.0)'}

def clean_text(text):
    #Remove references like [1], [2], etc..
    content = re.sub(r'\[\d+\]', '', text)
    content = re.sub(r'[^\w\s\.]', '', content)  # Remove punctuation (except periods)
    return content

def fetch_and_clean(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        for tag in soup(['script', 'style']):
            tag.decompose()
        
        main_content = None
        main_content = soup.find('div', {'class': 'mw-parser-output'})
        if not main_content:
            main_content = soup.find('div', {'class': 'content'})
        if not main_content:
            main_content = soup.find('div', {'class': 'article-content'}) or \
                          soup.find('div', {'class': 'post-content'}) or \
                          soup.find('main') or \
                          soup.find('article')
        
        if not main_content:
            main_content = soup.find('body')
        if not main_content:
            print(f"Warning: could not find main content for {url}, skipping.")
            return ''
        for section_title in ['References', 'Bibliography', 'External links', 'See also', 'Notes', 'Related links']:
            for section in main_content.find_all('span', id=section_title):
                for sib in section.parent.find_next_siblings():
                    sib.decompose()
                section.parent.decompose()
        
        text = main_content.get_text(separator=' ', strip=True)
        text = clean_text(text)
        return text if text else ''
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return ''

Each project will reuqire specific names and paths from original data

In [ ]:
docs_dir = data_dir / 'documents'
docs_dir.mkdir(exist_ok=True)
def download_docs(urls):
    for url in urls:
        clean_text = fetch_and_clean(url)
        if clean_text is None:
            print(f"No content for {url}, skipping.")
            continue
        doc_split = url.split('/')
        doc_name = doc_split[-1] if doc_split[-1] != '' else doc_split[-2]
        filename = os.path.join(docs_dir, doc_name + '.txt')
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(clean_text)
    
download_docs(urls)


2026-03-25 12:28:55,380 - WARNING - Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


Error fetching https://www.faa.gov/uas/: 403 Client Error: Forbidden for url: https://www.faa.gov/uas/


In [ ]:
docs = SimpleDirectoryReader(docs_dir).load_data()
docs[0]

Document(id_='f737c2ac-2dab-4346-a746-04f6a95d80d1', embedding=None, metadata={'file_path': 'c:\\Users\\dorot\\OneDrive - Politecnico di Torino\\Desktop\\Progetti\\RAG_Books\\RAG Driven Generative AI\\Ch3\\data\\documents\\1804.06985.txt', 'file_name': '1804.06985.txt', 'file_type': 'text/plain', 'file_size': 3782, 'creation_date': '2026-03-25', 'last_modified_date': '2026-03-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='High Energy Physics  Theory arXiv1804.06985 hepth Submitted on 19 Apr 2018 Title A Near Horizon Extreme Binary Black Hole Geometry Authors Jacob Ciafre  Maria J. Rodriguez View a PDF of the paper titled A Near Hori

The SimplyDirectoryReader is designed for working with unstructured data. It recursively scans teh directory and identifies and loads all supported file types (txt, docx...). It then extracts teh content from each file and returns a list of document objects with its text and metadata, such as teh filename and filepath. 

**Pipeline 2: creating and populating a Deep lake vector store**

We will create a Deep Lake vector store and populate it with data in our documents. We will implement a standard tensor configuration with:
- text(str): the text is the content of one of the text files listed in the dictionary of documents.
- metadata(json): will contain the filename source of each chunk of text for full transparency and control.
- embedding(float32)
- id(str, auto-populated): a unique id is attributed automatically to each chunk. the vector store will also contain an index, which is  a number from 0 to n, but it cannot be used semantically since it will change each time we modify the dataset. However, the id field will remain unchanged until we decide to optimize it with index-based search strategies.

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

2026-03-25 12:29:15,608 - INFO - Use pytorch device_name: cuda:0
2026-03-25 12:29:15,609 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


In [ ]:
import deeplake
import shutil
import time
from pathlib import Path
from llama_index.core import StorageContext
from llama_index.vector_stores.deeplake import DeepLakeVectorStore
from llama_index.core import VectorStoreIndex

vector_store_path = data_dir / "dory_vector_store"
vector_store_path.mkdir(parents=True, exist_ok=True)

def ensure_deeplake_dataset(path: Path):
    try:
        ds = deeplake.load(str(path))
        print("Loaded existing Deep Lake dataset:", path)
        return ds
    except Exception as e:
        print("deeplake.load failed:", type(e).__name__, e)
        meta_file = path / "dataset_meta.json"
        lock_file = path / "dataset_lock.lock"
        # Try removing a stale lock then reload
        if lock_file.exists():
            try:
                lock_file.unlink()
                print("Removed stale lock file:", lock_file)
            except Exception as e_lock:
                print("Failed removing lock:", e_lock)
        if not meta_file.exists():
            # Backup partial dataset and create a fresh empty dataset
            backup_path = str(path) + "_backup_" + str(int(time.time()))
            print("Backing up partial dataset to:", backup_path)
            shutil.move(str(path), backup_path)
            ds = deeplake.empty(str(path))
            print("Created new empty Deep Lake dataset at:", path)
            return ds
        # If meta exists but load still failed, re-raise
        raise

try:
    ds = ensure_deeplake_dataset(vector_store_path)
    # Initialize LlamaIndex DeepLake vector store using the dataset path
    vector_store = DeepLakeVectorStore(dataset_path=str(vector_store_path), overwrite=False)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(documents=docs, storage_context=storage_context, embed_model=model)
    print("Deep Lake vector store and index initialized successfully.")
except Exception as e:
    print("Failed to initialize Deep Lake vector store/index:", type(e).__name__, e)
    raise

deeplake.load failed: DatasetCorruptError Exception occurred (see Traceback). The dataset maybe corrupted. Try using `reset=True` to reset HEAD changes and load the previous commit. This will delete all uncommitted changes on the branch you are trying to load.
Backing up partial dataset to: c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch3\data\dory_vector_store_backup_1774438349
c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch3\data\dory_vector_store loaded successfully.


Created new empty Deep Lake dataset at: c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\RAG Driven Generative AI\Ch3\data\dory_vector_store
Failed to initialize Deep Lake vector store/index: ValueError At least one embedding tensor should exist.


ValueError: At least one embedding tensor should exist.

In [ ]:
vector_store = DeepLakeVectorStore(dataset_path=dataset_path, overwrite=True)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

NameError: name 'documents' is not defined

The index created will be reorganized by the indexing methods, which will rearrange and create new indexes when necessary.

In [ ]:
import deeplake
ds = deeplake.load(dataset_path)

./indexsearchrag loaded successfully.



In [ ]:
import json
import pandas as pd
import numpy as np

data = {}

for tensor_name in ds.tensors:
    tensor_data = ds[tensor_name].numpy()
    if tensor_data.ndim > 1:
        data[tensor_name] = [np.array(e).flatten().tolist() for e in tensor_data]
    else:
        if tensor_name == 'text':
            data[tensor_name] = [t.tobytes().decode('utf-8') for t in tensor_data]
        else:
            data[tensor_name] = tensor_data.tolist()
    
df = pd.DataFrame(data)

In [ ]:
def display_record(record_number):
    record = df.iloc[record_number]
    display_data = {
        "ID": record.get("id", "N/A"),
        "Metadata": record.get("metadata", "N/A"),
        "Text": record.get("text", "N/A"),
        "Embedding": record.get("embedding", "N/A")
    }
    print("ID:")
    print(display_data["ID"])
    print()
    print("Metadata:")
    metadata = display_data["Metadata"]
    if isinstance(metadata, list):
        for item in metadata:
            for key, value in item.items():
                print(f"{key}: {value}")
            print()
    else:
        print(metadata)
    print()
    print("Text:")
    print(display_data["Text"])
    print()
    print("Embedding:")
    print(display_data["Embedding"])
    print()

rec = 0 
display_record(rec)

The id is the index we will be using to organize teh chunks of text of teh text colum in the dataset. The chunks will be transformed into nodes that can contain teh original text, summaries and additional information. 

The metadata was automatically generated by SimpleDirectoryReader.

The text column contains the chunked data.

The embedding of each chunk of datawas generated through an embedding model that we do not need to configure.

**Pipeline 3: Index-based RAG**

We will implement four index engines:
- Vector Store index Engine: creates a vector store index from the documents, enabling efficient similarity-based searches.
- Tree Index: builds a hierarchical tree index from teh documents.
- List Index: constructs a straightfroward idnex from the documents.
- Keyword table Index: creates an index based on keywords extracted from the documents.

We will implementing querying with an LLM: queries the index with user input, retrieves teh relevanmt documents and returns a synthesized response.

In [ ]:
user_input = "What is the VisDrone dataset?"

In [ ]:
k = 3 #number of retrieved documents to use for the generation of the answer
temp = 0.1 #temperature for the generation of the answer
mt = 1024 #number of tokens to use for the retrieved context when generating the answer

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_similarity(text1, text2):
    emb1 = model.encode([text1])
    emb2 = model.encode([text2])
    similarity = cosine_similarity([emb1], [emb2])
    return similarity[0][0]

2026-03-24 18:59:05,598 - INFO - Use pytorch device_name: cuda:0
2026-03-24 18:59:05,599 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


1) Vector Store Index Query Engine
VectorStoreIndex is a type of index within LlamaIndex that implements vector embeddings to represent and retrieve infromation from docs. These docs with similar meanings will have embeddings that are closer together in teh vector space. However, this time, it does not use the existing Deep Lake vector store, it can create a new memory vector index, re-embed the docs and create a new index structure. 

In [ ]:
from llama_index.core import VectorStoreIndex
vector_store_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(type(vector_store_index))

2026-03-24 19:05:05,379 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,381 - INFO - Retrying request to /embeddings in 0.401726 seconds
2026-03-24 19:05:05,870 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,870 - INFO - Retrying request to /embeddings in 0.855444 seconds
2026-03-24 19:05:06,813 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:06,815 - INFO - Retrying request to /embeddings in 1.555540 seconds
2026-03-24 19:05:08,468 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:08,469 - INFO - Retrying request to /embeddings in 3.368408 seconds
2026-03-24 19:05:11,893 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:11,895 - INFO - Retrying request 

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
vector_query_engine = vector_store_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

NameError: name 'vector_store_index' is not defined

This function executes a query using a vector query engine and processes teh results into a structured format.

In [ ]:
import pandas as pd
import textwrap

def index_query(input_query):
    response = vector_query_engine.query(input_query)
    print(textwrap.fill(str(response), width=80))
    node_data =[]
    for node in response.source_nodes:
        noe_info = {
            "id": node.node_id,
            "text": node.node.get_text(),
            "score" : node.score
        }
        node_data.append(noe_info)
    df = pd.DataFrame(node_data)
    return df, response

Optimized chunking: we can decide the chunk size or teh code determines teh chunk size automatically.

In [ ]:
for node in response.source_nodes:
    node = node.node
    chunk_size = len(node.get_text())
    print(f"Node ID: {node.node_id}, Chunk Size: {chunk_size}")


We first calculate the sum of the scores and the average scores, and then we divide the weighted average by the time elapsed to perform the query. This metric is not an absolute value. It's an indicator that we can use to compare this engine with others.



In [ ]:
import numpy as np

def info_metrics(response):
    scores = [node.score for node in response.source_nodes]
    if scores:
        weights = np.exp(scores) / np.sum(np.exp(scores))  # Softmax to get weights
        perf = np.average(scores, weights=weights) / elapsed_time  # Example performance metric 
    else:
        perf = 0
    return perf

2) Tree index query Engine
The tree index organizes docs in a tree structure with broader summaries at higher levels and detailed information lower levels. Each node in the tree summarizes the text it covers. It is efficient for large datasets and queries large collections of docs rapisly by breaking them down into optimized chunks. In this model, the LLM acts like it is answering a multiple-choice question when selecting the best nodes during a query. It analyzes the query, compares it with teh summaries of the current node's children and decides which path to follow to find the most relevant information.

In [ ]:
from llama_index.core import TreeIndex
tree_index = TreeIndex.from_documents(documents)   

In [ ]:
import time
import textwrap

start_time = time.time()
response = tree_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

NameError: name 'tree_query_engine' is not defined

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

3) Last index query engine
The query engine will process the user input and each doc as a prompt for an LLM. The LLm will evaluate the semantic similarity relationship between the docs and the query, thus implicity ranking and selecting teh most relavnt nodes. The selection with an LLM is not rule-based. Nothing is predefined, which means that the selction is prompt-based by combining the user input with a collection of docs. The LLM evaluates each doc in teh list indipendently, assigning a score based on its perceived relevance to teh query. Then, teh top-k docs are retained by the query engine.

In [ ]:
from llam_index.core import ListIndex
list_index = ListIndex.from_documents(documents)

In [ ]:
list_query_engine = list_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

In [ ]:
start_time = time.time()
response = list_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

4) Keyword index query engine
It is designed to extract keywords from your docs and organize them in a table-like structure. This strcuture makes it easier to query and retrieve relevant information based on specific keyowrds or topics.

In [ ]:
from llama_index.core import KeywordTableIndex  
keyword_index = KeywordTableIndex.from_documents(documents)

2026-03-24 19:52:15,799 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:15,803 - INFO - Retrying request to /chat/completions in 0.444031 seconds
2026-03-24 19:52:16,986 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:16,988 - INFO - Retrying request to /chat/completions in 0.867025 seconds
2026-03-24 19:52:18,546 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:18,548 - INFO - Retrying request to /chat/completions in 1.589668 seconds
2026-03-24 19:52:20,861 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:20,866 - WARNING - Retrying llama_index.llms.openai.base.OpenAI._chat in 1 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
data = []
for keyword, doc_ids in keyword_index.keyword_to_doc_ids.items():
    for doc_id in doc_ids:
        data.append({"keyword": keyword, "doc_id": doc_id})

df = pd.DataFrame(data)
df

The index created will be reorganized by the indexing methods, which will rearrange and create new indexes when necessary.

In [33]:
import deeplake
ds = deeplake.load(dataset_path)

./indexsearchrag loaded successfully.



In [34]:
import json
import pandas as pd
import numpy as np

data = {}

for tensor_name in ds.tensors:
    tensor_data = ds[tensor_name].numpy()
    if tensor_data.ndim > 1:
        data[tensor_name] = [np.array(e).flatten().tolist() for e in tensor_data]
    else:
        if tensor_name == 'text':
            data[tensor_name] = [t.tobytes().decode('utf-8') for t in tensor_data]
        else:
            data[tensor_name] = tensor_data.tolist()
    
df = pd.DataFrame(data)

In [35]:
def display_record(record_number):
    record = df.iloc[record_number]
    display_data = {
        "ID": record['id'] if 'id' in record else None,
        "Text": record['text'] if 'text' in record else None,
        "Metadata": record['metadata'] if 'metadata' in record else None,
        "Embedding": record['embedding'] if 'embedding' in record else None,
    }

In [36]:
rec = 0
display_record(rec)

IndexError: single positional indexer is out-of-bounds

The id is the index we will be using to organize teh chunks of text of teh text colum in the dataset. The chunks will be transformed into nodes that can contain teh original text, summaries and additional information. 

The metadata was automatically generated by SimpleDirectoryReader.

The text column contains the chunked data.

The embedding of each chunk of datawas generated through an embedding model that we do not need to configure.

**Pipeline 3: Index-based RAG**

We will implement four index engines:
- Vector Store index Engine: creates a vector store index from the documents, enabling efficient similarity-based searches.
- Tree Index: builds a hierarchical tree index from teh documents.
- List Index: constructs a straightfroward idnex from the documents.
- Keyword table Index: creates an index based on keywords extracted from the documents.

We will implementing querying with an LLM: queries the index with user input, retrieves teh relevanmt documents and returns a synthesized response.

In [ ]:
user_input = "What is the VisDrone dataset?"

In [38]:
k = 3 #number of retrieved documents to use for the generation of the answer
temp = 0.1 #temperature for the generation of the answer
mt = 1024 #number of tokens to use for the retrieved context when generating the answer

In [41]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_similarity(text1, text2):
    emb1 = model.encode([text1])
    emb2 = model.encode([text2])
    similarity = cosine_similarity([emb1], [emb2])
    return similarity[0][0]

2026-03-24 18:59:05,598 - INFO - Use pytorch device_name: cuda:0
2026-03-24 18:59:05,599 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


1) Vector Store Index Query Engine
VectorStoreIndex is a type of index within LlamaIndex that implements vector embeddings to represent and retrieve infromation from docs. These docs with similar meanings will have embeddings that are closer together in teh vector space. However, this time, it does not use the existing Deep Lake vector store, it can create a new memory vector index, re-embed the docs and create a new index structure. 

In [45]:
from llama_index.core import VectorStoreIndex
vector_store_index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(type(vector_store_index))

2026-03-24 19:05:05,379 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,381 - INFO - Retrying request to /embeddings in 0.401726 seconds
2026-03-24 19:05:05,870 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:05,870 - INFO - Retrying request to /embeddings in 0.855444 seconds
2026-03-24 19:05:06,813 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:06,815 - INFO - Retrying request to /embeddings in 1.555540 seconds
2026-03-24 19:05:08,468 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:08,469 - INFO - Retrying request to /embeddings in 3.368408 seconds
2026-03-24 19:05:11,893 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:05:11,895 - INFO - Retrying request 

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [44]:
vector_query_engine = vector_store_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

NameError: name 'vector_store_index' is not defined

This function executes a query using a vector query engine and processes teh results into a structured format.

In [ ]:
import pandas as pd
import textwrap

def index_query(input_query):
    response = vector_query_engine.query(input_query)
    print(textwrap.fill(str(response), width=80))
    node_data =[]
    for node in response.source_nodes:
        noe_info = {
            "id": node.node_id,
            "text": node.node.get_text(),
            "score" : node.score
        }
        node_data.append(noe_info)
    df = pd.DataFrame(node_data)
    return df, response

Optimized chunking: we can decide the chunk size or teh code determines teh chunk size automatically.

In [ ]:
for node in response.source_nodes:
    node = node.node
    chunk_size = len(node.get_text())
    print(f"Node ID: {node.node_id}, Chunk Size: {chunk_size}")


We first calculate the sum of the scores and the average scores, and then we divide the weighted average by the time elapsed to perform the query. This metric is not an absolute value. It's an indicator that we can use to compare this engine with others.



In [ ]:
import numpy as np

def info_metrics(response):
    scores = [node.score for node in response.source_nodes]
    if scores:
        weights = np.exp(scores) / np.sum(np.exp(scores))  # Softmax to get weights
        perf = np.average(scores, weights=weights) / elapsed_time  # Example performance metric 
    else:
        perf = 0
    return perf

2) Tree index query Engine
The tree index organizes docs in a tree structure with broader summaries at higher levels and detailed information lower levels. Each node in the tree summarizes the text it covers. It is efficient for large datasets and queries large collections of docs rapisly by breaking them down into optimized chunks. In this model, the LLM acts like it is answering a multiple-choice question when selecting the best nodes during a query. It analyzes the query, compares it with teh summaries of the current node's children and decides which path to follow to find the most relevant information.

In [46]:
from llama_index.core import TreeIndex
tree_index = TreeIndex.from_documents(documents)   

In [49]:
import time
import textwrap

start_time = time.time()
response = tree_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

NameError: name 'tree_query_engine' is not defined

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

3) Last index query engine
The query engine will process the user input and each doc as a prompt for an LLM. The LLm will evaluate the semantic similarity relationship between the docs and the query, thus implicity ranking and selecting teh most relavnt nodes. The selection with an LLM is not rule-based. Nothing is predefined, which means that the selction is prompt-based by combining the user input with a collection of docs. The LLM evaluates each doc in teh list indipendently, assigning a score based on its perceived relevance to teh query. Then, teh top-k docs are retained by the query engine.

In [ ]:
from llam_index.core import ListIndex
list_index = ListIndex.from_documents(documents)

In [ ]:
list_query_engine = list_index.as_query_engine(similarity_top_k=k, temperature=temp, max_tokens=mt)

In [ ]:
start_time = time.time()
response = list_query_engine.query(user_input)
end_time = time.time()
elapsed_time = end_time - start_time

In [ ]:
similarity_score = calculate_similarity(user_input, response.get_text())
performance = similarity_score / elapsed_time if elapsed_time > 0 else 0

4) Keyword index query engine
It is designed to extract keywords from your docs and organize them in a table-like structure. This strcuture makes it easier to query and retrieve relevant information based on specific keyowrds or topics.

In [50]:
from llama_index.core import KeywordTableIndex  
keyword_index = KeywordTableIndex.from_documents(documents)

2026-03-24 19:52:15,799 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:15,803 - INFO - Retrying request to /chat/completions in 0.444031 seconds
2026-03-24 19:52:16,986 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:16,988 - INFO - Retrying request to /chat/completions in 0.867025 seconds
2026-03-24 19:52:18,546 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:18,548 - INFO - Retrying request to /chat/completions in 1.589668 seconds
2026-03-24 19:52:20,861 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
2026-03-24 19:52:20,866 - WARNING - Retrying llama_index.llms.openai.base.OpenAI._chat in 1 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
data = []
for keyword, doc_ids in keyword_index.keyword_to_doc_ids.items():
    for doc_id in doc_ids:
        data.append({"keyword": keyword, "doc_id": doc_id})

df = pd.DataFrame(data)
df